# Réutiliser des modèles pré-entraînés

Dans cette séance, vous aller réutiliser des modèles pré-entraînés pour des tâches de classification et de génération d'images.
Ces modèles sont disponibles dans la librairie `transformers` de HuggingFace, qui fournit une interface simple pour les utiliser et les adapter à vos besoins.


In [ ]:
# !pip install -q torch torchvision transformers datasets diffusers accelerate

import glob

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

import matplotlib.pyplot as plt
from PIL import Image

from transformers import AutoImageProcessor, AutoModel, pipeline
from diffusers import DDPMPipeline

from training_toolbox import Trainer, freeze, unfreeze, count_trainable_parameters, accuracy

torch.manual_seed(0)


## Partie 1 — Un modèle pré-entraîné, sans ré-entraînement

La librairie `transformers` fournit une interface simplifiée pour réutiliser des modèles pré-entraînés. Dans le code ci-dessous, vous allez charger un modèle
`ResNet-50` entraîné sur ImageNet (1000 classes) — un type d'architecture que vous avez déjà étudié, mais entraîné sur bien plus de données que ce que l'on peut faire en
TP. On ne fait **aucun** entraînement, juste de l'inférence, sur quelques images que vous aurez préalablement enregistrées dans le dossier courant.


In [ ]:
classifier = pipeline("image-classification", model="microsoft/resnet-50")

# TODO : ci-dessous une liste de chemins vers des fichiers image
sample_paths = [
    "cats_and_dogs/train_catdog/cat.0.jpg",
    "cats_and_dogs/train_catdog/dog.0.jpg",
]

fig, axes = plt.subplots(1, len(sample_paths), figsize=(4 * len(sample_paths), 4))
for ax, path in zip(axes, sample_paths):
    img = Image.open(path).convert("RGB")
    preds = classifier(img)
    ax.imshow(img)
    ax.axis("off")
    title = "\n".join(f"{p['label']} ({p['score']:.2f})" for p in preds[:3])
    ax.set_title(title, fontsize=9)
plt.tight_layout()
plt.show()


**Questions.**

- ImageNet ne contient pas de classe générique "chat" ou "chien", mais des races précises
  (`tabby cat`, `Labrador retriever`...). Le modèle s'en sort-il malgré tout pour distinguer
  chat et chien sur vos exemples ?
- Le `pipeline` renvoie un score de confiance par classe : que représente-t-il précisément ?
- Essayez une image qui n'a clairement rien à voir avec les classes ImageNet (un objet du
  quotidien insolite, une capture d'écran...). Que se passe-t-il ?

## Partie 2 — Sous le capot : `AutoImageProcessor` + `AutoModel`

Un `pipeline` masque deux objets : un **processor** (image → tenseur normalisé, de la bonne
taille) et un **modèle** (tenseur → représentation, puis logits si le modèle a une tête de
classification). Regardons-les séparément, avec `AutoModel` (le backbone **sans** tête de
classification, contrairement à `AutoModelForImageClassification` utilisé implicitement par
le `pipeline` ci-dessus).


In [ ]:
processor = AutoImageProcessor.from_pretrained("microsoft/resnet-50")
backbone = AutoModel.from_pretrained("microsoft/resnet-50")

img = Image.open(sample_paths[0]).convert("RGB")
inputs = processor(images=img, return_tensors="pt")
print("Clés produites par le processor :", list(inputs.keys()))
print("Shape de pixel_values :", inputs["pixel_values"].shape)

with torch.no_grad():
    output = backbone(**inputs)

print("last_hidden_state :", output.last_hidden_state.shape)  # carte de features avant pooling
print("pooler_output      :", output.pooler_output.shape)      # vecteur global de l'image


`microsoft/resnet-50` sait produire une représentation vectorielle d'une image
(`pooler_output`, de dimension `backbone.config.hidden_sizes[-1]`), mais ne sait pas encore
classer un chat ou un chien : il n'a pas de tête de classification pour cette tâche
spécifique.

## Partie 3 — Backbone gelé + tête entraînée

On charge le dataset `cats_and_dogs` :


In [ ]:
class CatsAndDogsDataset(Dataset):
    """Fichiers `cat.<id>.jpg` / `dog.<id>.jpg` à plat dans `folder` (label déduit du
    nom de fichier). `transform` doit renvoyer un tenseur `pixel_values` déjà prétraité
    (on réutilise directement le `processor` Hugging Face comme transform)."""

    def __init__(self, folder, transform):
        self.paths = sorted(glob.glob(f"{folder}/*.jpg"))
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        label = 0 if "cat" in path.split("/")[-1] else 1
        img = Image.open(path).convert("RGB")
        pixel_values = self.transform(img)
        return pixel_values, label


def hf_transform(img):
    # Renvoie directement un tenseur (C, H, W) déjà normalisé/redimensionné comme attendu
    # par le backbone -- le Trainer le traite ensuite comme une image "classique" (x, y).
    return processor(images=img, return_tensors="pt")["pixel_values"][0]


train_data = CatsAndDogsDataset("./cats_and_dogs/train_catdog", transform=hf_transform)
test_data = CatsAndDogsDataset("./cats_and_dogs/test_catdog", transform=hf_transform)

# Note : le dataset ne fournit qu'un train et un test (pas de val dédiée) -- on réutilise le
# test set comme validation ici, par souci de simplicité pour ce TP. Dans un vrai projet, on
# préférerait un découpage train/val/test à trois blocs distincts.
train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
val_loader = DataLoader(test_data, batch_size=32)

print(f"Train : {len(train_data)} images - Val/test : {len(test_data)} images")


**Question 3.1.** Implémentez un modèle `ImageClassifier` qui prend en entrée un backbone pré-entraîné (ici `microsoft/resnet-50`) et y ajoute une tête de classification linéaire pour classer les images en deux classes : chat ou chien. Le backbone doit être **gelé** (ses poids ne doivent pas être mis à jour pendant l'entraînement) à l'aide de la fonction `freeze()` qui prend comme argument un `nn.Module`. 
Combien le modèle résultant a-t-il de paramètres entrainables ?
Entraînez la tête de classification sur le dataset `cats_and_dogs` pendant 3 epochs et évaluez la performance du modèle sur le jeu de test.

In [ ]:
class ImageClassifier(nn.Module):
    """Backbone pré-entraîné + une tête linéaire pour la classification binaire chat/chien."""

    def __init__(self, backbone, n_classes=2):
        super().__init__()
        # TODO

    def forward(self, pixel_values):
        # TODO
        pass


# TODO : instancier ImageClassifier(backbone), geler le backbone (freeze(model.backbone)),
#        afficher le nombre de paramètres entraînables (count_trainable_parameters(model)),
#        puis entraîner la tête avec le Trainer (Adam sur model.head.parameters(),
#        nn.CrossEntropyLoss(), metrics={"acc": accuracy}) pendant 3 epochs -> history_frozen

## Partie 4 — Dégeler quelques couches

Le backbone gelé sert de simple extracteur de features génériques (apprises sur ImageNet).
En dégelant les dernières couches (celles qui encodent les notions les plus spécifiques, par
opposition aux premières couches qui détectent des motifs très génériques comme des
contours), on peut souvent gagner en performance, au prix d'un entraînement plus coûteux.

**Question 4.1.** `backbone.encoder.stages` est la liste des "étages" du ResNet
(inspectez sa longueur et sa structure avec `print(backbone.encoder.stages)` si besoin).
Dégelez le **dernier** étage uniquement. Combien le modèle résultant a-t-il de paramètres entrainables ? Entraînez le modèle sur le dataset `cats_and_dogs` pendant 3 epochs supplémentaires avec un taux d'apprentissage réduit ($10^-5$) et évaluez la performance du modèle sur le jeu de test. Comparez avec la performance obtenue avec le backbone gelé.


In [ ]:
print(f"Nombre d'étages du backbone : {len(model.backbone.encoder.stages)}")

# TODO : dégeler le dernier étage (unfreeze(model.backbone.encoder.stages[-1])), afficher le
#        nombre de paramètres entraînables, construire un optimizer Adam (lr=1e-5) sur les
#        seuls paramètres avec requires_grad=True, un Trainer, et entraîner 2 epochs
#        -> history_finetuned

**Questions.**

- Comparez `history_frozen` et `history_finetuned` (accuracy de validation). Le gain observé
  justifie-t-il le coût de calcul supplémentaire ?
- Comparez cette approche à celle d'un TP précédent où vous utilisiez un modèle appris _from scratch_ : une méthode semble-t-elle plus efficace que l'autre ? Pourquoi ?
- Que se passerait-il, à votre avis, si on dégelait *tout* le backbone d'un coup avec un jeu
  d'entraînement aussi petit (quelques centaines d'images) ?

## Partie 5 — Réutiliser un modèle génératif pré-entraîné

Vous allez maintenant charger un modèle de **Conditional Flow Matching**
déjà entraîné, et l'utiliser uniquement en inférence. 
Le principe est exactement celui de votre `VelocityField` de la séance 10 (un
réseau qui prédit un champ de vitesses, échantillonné en intégrant une ODE de $t=0$ à $t=1$),
mais appliqué ici à de vraies images 32x32 plutôt qu'à un nuage de points 2D. 

**Question 5.1.**  chargez le modèle `FrankCCCCC/cfm-cifar10-32` et générez 4 nouvelles images (vous utiliserez un nombres faibles de pas d'intégration, par exemple 50).

In [ ]:
# TODO : charger le pipeline DDPMPipeline.from_pretrained("FrankCCCCC/cfm-cifar10-32"),
#        générer 4 images (num_inference_steps=50 par exemple), et les afficher côte à côte

**Questions.**

- Contrairement à votre `VelocityField` de la séance 10 (entrée 2D+temps, quelques centaines
  de milliers de paramètres, entraîné en quelques secondes), ce modèle traite des images
  32x32 réelles. Qu'est-ce qui a dû changer dans l'architecture du réseau pour passer d'un
  problème à l'autre ?
- Essayez de réduire `num_inference_steps` (par exemple à 5 ou 10). Que se passe-t-il sur la
  qualité des images générées ? Comparez à ce que vous aviez observé séance 10 en réduisant le
  nombre de pas d'Euler pour le modèle 2D.
- Qu'apporte concrètement la réutilisation d'un modèle pré-entraîné ici ?

## Pour aller plus loin (optionnel, coûteux en calcul)

- `StableDiffusionPipeline` (modèle `runwayml/stable-diffusion-v1-5`) permet de générer une
  image à partir d'un prompt texte : le principe est le même que ci-dessus, mais le
  débruitage est *conditionné* par une représentation du texte. Beaucoup plus lourd (à
  réserver à un GPU, hors séance) :

```python
from diffusers import StableDiffusionPipeline
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5", torch_dtype=torch.float16
)
pipe = pipe.to("cuda")
image = pipe("a photo of a computer science teacher surfing a gigantic wave").images[0]
```

- Remplacer `microsoft/resnet-50` par un ViT (`google/vit-base-patch16-224`) en Partie 1-3 :
  même démarche, architecture différente (transformer plutôt que convolutif) — l'occasion de
  vérifier que l'API Hugging Face reste identique d'un type d'architecture à l'autre.
- Utiliser `EarlyStopping` / `ModelCheckpoint` pendant le
  fine-tuning de la Partie 3-4.